# صدى | Sada Reviews — CAMeLBERT Fine-tuning

دفتر Colab لتدريب CAMeLBERT على تصنيف موضوع مراجعة التطبيق. يتضمن البيانات المصطنعة من المشروع: 7 فئات و7 شرائح لغوية.

**تنبيه منهجي:** CAMeLBERT نموذج عربي. نقيس الإنجليزية وArabizi منفصلًا، ولا نفترض تكافؤ الأداء. البيانات الحالية مصطنعة؛ النتائج للتجربة التعليمية فقط.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas matplotlib


## 1. ارفع بيانات المشروع

من GitHub نزّل `data/sample_reviews.csv` ثم ارفعه إلى جلسة Colab. يحتوي الأعمدة review وcategory وsentiment وlanguage وdialect وsplit.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_name = next(name for name in uploaded if name.lower().endswith(".csv"))


In [ ]:
import inspect
import json
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv(csv_name)
required = {"review", "category", "language", "dialect", "split"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

def normalize_review(text):
    text = str(text).strip()
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " [EMAIL] ", text)
    text = re.sub(r"(?<!\d)(?:05\d{8}|\+9665\d{8})(?!\d)", " [PHONE] ", text)
    return re.sub(r"\s+", " ", text).strip()

df["review"] = df["review"].map(normalize_review)
train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)
labels = sorted(df["category"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
train_df["label"] = train_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)
print("Rows:", len(df), "| Train:", len(train_df), "| Test:", len(test_df))
print("Labels:", label2id)
print("Train label counts:\n", train_df["category"].value_counts())

## 2. تحميل CAMeLBERT وتجهيز الرموز

سنستخدم CAMeLBERT-Mix لتصنيف فئة المراجعة. يظل الاختبار حسب اللهجة مهمًا، خصوصًا للإنجليزية وArabizi.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

model_name = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(labels), label2id=label2id, id2label=id2label
)

train_hf = Dataset.from_pandas(train_df[["review", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["review", "label"]], preserve_index=False)
def tokenize_batch(batch):
    return tokenizer(batch["review"], truncation=True, max_length=128)
train_hf = train_hf.map(tokenize_batch, batched=True, remove_columns=["review"])
test_hf = test_hf.map(tokenize_batch, batched=True, remove_columns=["review"])
train_hf = train_hf.rename_column("label", "labels")
test_hf = test_hf.rename_column("label", "labels")
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized train/test:", len(train_hf), len(test_hf))

## 3. الضبط الدقيق

نستخدم 3 epochs كبداية، ثم نختار أفضل checkpoint وفق Macro-F1 على مجموعة الاختبار. مجموعة الاختبار هنا مبنية من صيغ بديلة مولدة؛ لا تمثل مراجعات متجر حقيقية.

In [ ]:
def compute_metrics(eval_pred):
    logits, gold = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(gold, pred),
        "macro_f1": f1_score(gold, pred, average="macro", zero_division=0),
    }

arg_names = inspect.signature(TrainingArguments).parameters
eval_key = "eval_strategy" if "eval_strategy" in arg_names else "evaluation_strategy"
training_kwargs = {
    "output_dir": "./sada_camelbert_checkpoints",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
}
training_kwargs[eval_key] = "epoch"
args = TrainingArguments(**training_kwargs)
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_hf, eval_dataset=test_hf,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

## 4. التقييم وتحليل اللهجات

نعرض Macro-F1 وAccuracy وتقرير الفئات، ثم نجزّئ Macro-F1 حسب اللهجة واللغة.

In [ ]:
test_df["predicted"] = [id2label[int(i)] for i in y_pred]
test_df["correct"] = test_df["category"] == test_df["predicted"]
dialect_rows = []
for (dialect, language), part in test_df.groupby(["dialect", "language"], dropna=False):
    dialect_rows.append({
        "dialect": dialect, "language": language, "n": len(part),
        "accuracy": accuracy_score(part["category"], part["predicted"]),
        "macro_f1": f1_score(part["category"], part["predicted"], average="macro", labels=labels, zero_division=0),
    })
dialect_report = pd.DataFrame(dialect_rows)
display(dialect_report.sort_values("macro_f1"))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ConfusionMatrixDisplay.from_predictions(
    test_df["category"], test_df["predicted"], labels=labels,
    display_labels=labels, xticks_rotation=45, cmap="Blues", ax=axes[0]
)
axes[0].set_title("CAMeLBERT — Confusion Matrix")
dialect_report.sort_values("macro_f1").plot(
    kind="barh", x="dialect", y="macro_f1", legend=False,
    color="#2878a5", ax=axes[1]
)
axes[1].set_xlim(0, 1)
axes[1].set_title("Macro-F1 by dialect / language")
axes[1].set_xlabel("Macro-F1")
plt.tight_layout()
plt.savefig("sada_camelbert_evaluation.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. حفظ النموذج والنتائج

احفظ أوزان النموذج بعد التدريب للاستخدام في الديمو. لا ترفع بيانات حقيقية تحتوي معلومات شخصية.

In [ ]:
output_dir = "./sada_camelbert_finetuned"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
test_df.to_csv("sada_camelbert_test_predictions.csv", index=False)
print("Saved:", output_dir)
files.download("sada_camelbert_evaluation.png")
files.download("sada_camelbert_test_predictions.csv")

## قراءة النتيجة

لا نعتبر دقة مرتفعة على هذه البيانات دليلًا كافيًا على نجاح حقيقي: العينة مولدة ومنظمة، والاختبار ليس من متجر فعلي. ركّز في العرض على الفئات التي يخلط بينها النموذج، والفجوة بين اللهجات، والحاجة إلى بيانات حقيقية ومراجعة بشرية.